In [ ]:
import folium
from folium.plugins import Fullscreen
import geopandas as gpd
import pandas as pd


In [ ]:
import black
import jupyter_black

jupyter_black.load(
    # lab=False,
    line_length=78,
    # verbosity="DEBUG",
    target_version=black.TargetVersion.PY310,
)

In [ ]:
gdf = gpd.read_file("data/own/bike-color-routes.geojson")

In [ ]:
def gen_map_colors(value):
    return {
        "Red": "DarkRed",
        "Orange": "#d95f02",  # orange
        "Yellow": "#e6ab02",  # goldenrod
        "Brown": "#a6761d",  # brown
        "Pink": "#e7298a",  # magenta
        "Gray": "#777777",  # neutral dark gray
        # "": "#b1b1b1",  # light medium gray
        "": "#9ecae1",  # subtle bluish tone
    }.get(value, value)

In [ ]:
gdf.loc[gdf["Route"] == "15th Ave", "Color"] = "Red"

In [ ]:
gdf["map_color"] = gdf["Color"].apply(gen_map_colors)

In [ ]:
def color_by_bike_segid(bike_segids, color):
    if isinstance(bike_segids, int):
        bike_segids = [bike_segids]
    for n in bike_segids:
        gdf.loc[gdf["bike_segid"] == n, "Color"] = color
    gdf["map_color"] = gdf["Color"].apply(gen_map_colors)


def color_by_routes(routes, color):
    if isinstance(routes, str):
        routes = [routes]

    for n in routes:
        gdf.loc[gdf["Route"] == n, "Color"] = color
    gdf["map_color"] = gdf["Color"].apply(gen_map_colors)

In [ ]:
major = gdf[gdf["Color"] != ""].copy()
minor = gdf[gdf["Color"] == ""].copy()

In [ ]:
m = minor.explore(
    tiles="Cartodb Positron",
    color=minor["map_color"],
    style_kwds=dict(weight=1, opacity=0.8),
    tooltip=["Route", "bike_segid"],
    zoom_start=13,
)

m = major.explore(
    m=m,
    tiles="Cartodb Positron",
    color=major["map_color"],
    style_kwds=dict(weight=5, opacity=0.9),
    tooltip=["Color", "Route", "bike_segid"],
)

title = "Proposed Eugene Major Bike Route Branding"
title_html = (
    f'<h1 style="position:absolute;z-index:100000;left:20vw" >{title}</h1>'
)
m.get_root().html.add_child(folium.Element(title_html))

help_html = """
<div style="
    position: fixed; 
    bottom: 40px;
    left: 40px;
    width: 260px;
    z-index:9999;
    background-color: white;
    border-radius: 8px;
    padding: 10px;
    box-shadow: 0 0 8px rgba(0,0,0,0.2);
    font-size:14px;
">
<p>
<a href="https://github.com/nbirnel/atlas-of-eugene">Source</a>
</p><br>
Hover over routes for details.<br>This map is scrollable and zoomable.
</div>
"""
Fullscreen(position="topright").add_to(m)
m.get_root().html.add_child(folium.Element(help_html))

m

In [ ]:
m.save("bike-color-routes.html")

In [ ]:
gdf.to_file("data/own/bike-color-routes.geojson")